# Speaker signatures: how the voices of the Gita differ

The Gita is a dialogue. Four voices speak it: **Krishna** (the teacher),
**Arjuna** (the student), **Sanjaya** (the narrator relaying the scene to the
blind king), and one opening verse from **Dhritarashtra** himself. Do they *talk*
differently, using a measurably different vocabulary and not only speaking about
different things?

This notebook builds a quantitative **signature** for each speaker, using only
`SPOKEN_BY` and the two vocabulary layers already in the graph, with no schema
change:

1. **Speaking share:** how much of the text each voice holds.
2. **Theme & concept fingerprints:** the topics each voice leans into, measured
   as *lift* over the whole text (so ubiquitous themes don't drown the signal).
3. **Keyness (Dunning log-likelihood):** the standard corpus-linguistics test
   for which words are statistically over-represented in one voice versus the
   rest. This is the scholarly core.
4. **Lexical profile:** content-word density and vocabulary spread per voice.

Read-only; every section ends in inline `assert`s. Figures export to `exports/`.

**A caveat kept honest:** the English `Term` layer is content lemmas (nouns and
verbs from spaCy), so "keyness" here means *content-word* keyness. Function
words are out of scope. Persisting POS would extend this (a separate task).

## 1. Setup & connect

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from graphdatascience import GraphDataScience

warnings.filterwarnings("ignore")


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))

import gita_kg as gk

EXPORTS = PKG / "exports"
EXPORTS.mkdir(exist_ok=True)

/Users/akhilesh.koul/Documents/GitHub/CodePlayground/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(PKG / ".env", override=True)
cfg = gk.load_config()
gds = GraphDataScience(cfg.uri, auth=(cfg.user, cfg.password), database=cfg.database)
gds.set_show_progress(False)
print("GDS server:", gds.version(), "| database:", cfg.database)

GDS server: 2026.7.0 | database: neo4j


In [3]:
def cypher(query: str, **params) -> pd.DataFrame:
    return gds.run_cypher(query, params)


def export(fig: go.Figure, filename: str) -> None:
    out = EXPORTS / filename
    fig.write_html(out, include_plotlyjs="cdn")
    print("wrote", out.relative_to(ROOT))


# Stable colour per speaker across every figure.
SPEAKER_COLOR = {
    "Krishna": "#1b7837", "Arjuna": "#762a83",
    "Sanjaya": "#2166ac", "Dhritarashtra": "#b2182b",
}

## 2. Speaking share

In [4]:
share = cypher(
    "MATCH (v:Verse)-[:SPOKEN_BY]->(p:Person) "
    "RETURN p.name AS speaker, count(*) AS verses ORDER BY verses DESC"
)
total_verses = cypher("MATCH (v:Verse) RETURN count(v) AS n").iloc[0]["n"]
assert int(share["verses"].sum()) == total_verses, int(share["verses"].sum())
share["pct"] = 100 * share["verses"] / total_verses
print(share.to_string(index=False))

# Speakers with enough verses for statistical comparison.
MIN_VERSES = 20
main_speakers = share.loc[share["verses"] >= MIN_VERSES, "speaker"].tolist()
print("\nspeakers compared statistically:", main_speakers)

      speaker  verses       pct
      Krishna     574 81.883024
       Arjuna      86 12.268188
      Sanjaya      40  5.706134
Dhritarashtra       1  0.142653

speakers compared statistically: ['Krishna', 'Arjuna', 'Sanjaya']


In [5]:
fig_share = px.bar(
    share, x="verses", y="speaker", orientation="h", text="pct",
    color="speaker", color_discrete_map=SPEAKER_COLOR,
    title="Who holds the floor: verses spoken by each voice",
    labels=dict(verses="verses spoken", speaker=""),
)
fig_share.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_share.update_layout(height=340, showlegend=False)
export(fig_share, "speaker_share.html")
fig_share

wrote gita-knowledge-graph/exports/speaker_share.html


## 3. Theme & concept fingerprints

For each voice, the share of its verse-weight going to each theme, then **lift** =
that share divided by the theme's share across the whole text. Lift > 1 means the
voice leans *into* a theme; < 1 means it avoids it.

In [6]:
def fingerprint(rel: str, node: str):
    df = cypher(
        f"MATCH (p:Person)<-[:SPOKEN_BY]-(v:Verse)-[m:{rel}]->(x:{node}) "
        f"RETURN p.name AS speaker, x.name AS item, sum(m.weight) AS w"
    )
    mat = df.pivot_table(index="speaker", columns="item", values="w", aggfunc="sum",
                         fill_value=0.0)
    mat = mat.loc[[s for s in main_speakers if s in mat.index]]
    share_mat = mat.div(mat.sum(axis=1), axis=0)
    global_share = mat.sum(axis=0) / mat.sum().sum()
    lift = share_mat.div(global_share, axis=1)
    # Each speaker's shares sum to 1.
    assert np.allclose(share_mat.sum(axis=1).values, 1.0)
    return share_mat, lift


theme_share, theme_lift = fingerprint("MENTIONS_THEME", "Theme")
concept_share, concept_lift = fingerprint("EXPRESSES_CONCEPT", "Concept")
for s in main_speakers:
    if s in theme_lift.index:
        top = theme_lift.loc[s].sort_values(ascending=False).head(3)
        print(f"{s} leans into:", ", ".join(f"{t} (×{v:.1f})" for t, v in top.items()))

Krishna leans into:

 moksha (×1.1), sacrifice-austerity (×1.1), guna (×1.1)
Arjuna leans into: dharma (×2.9), brahman (×2.3), samsara (×1.8)
Sanjaya leans into: dharma (×3.5), yoga (×2.0), bhakti (×1.7)


In [7]:
# Figure: theme fingerprint as a radar (share per speaker).
themes = list(theme_share.columns)
fig_radar = go.Figure()
for s in theme_share.index:
    vals = theme_share.loc[s, themes].tolist()
    fig_radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=themes + [themes[0]], name=s,
        line=dict(color=SPEAKER_COLOR.get(s)), fill="toself", opacity=0.55,
    ))
fig_radar.update_layout(
    title="Theme fingerprint: where each voice spends its attention",
    height=600, polar=dict(radialaxis=dict(visible=True)),
)
export(fig_radar, "speaker_theme_radar.html")
fig_radar

wrote gita-knowledge-graph/exports/speaker_theme_radar.html


In [8]:
# Figure: concept lift heatmap (Sanskrit-grounded, log2 so over/under-use is symmetric).
log_lift = np.log2(concept_lift.replace(0, np.nan))
fig_cfp = px.imshow(
    log_lift, aspect="auto", color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0.0,
    labels=dict(x="concept", y="speaker", color="log2 lift"),
    title="Concept fingerprint: over- (red) and under-use (blue) vs the whole text",
)
fig_cfp.update_layout(height=360)
export(fig_cfp, "speaker_concept_lift.html")
fig_cfp

wrote gita-knowledge-graph/exports/speaker_concept_lift.html


## 4. Keyness: Dunning log-likelihood

The standard test in corpus linguistics for *keywords*: compare each content
lemma's frequency in one voice (the target) against all the other voices (the
reference). The **log-likelihood** statistic G² flags words used far more (or
less) than chance. High positive G² with over-use = a signature word.

$$G^2 = 2\sum_i O_i \ln\!\frac{O_i}{E_i}$$

In [9]:
tbs = cypher(
    "MATCH (p:Person)<-[:SPOKEN_BY]-(v:Verse)-[m:MENTIONS_TERM]->(t:Term) "
    "RETURN p.name AS speaker, t.lemma AS term, sum(m.count) AS n"
)
counts = tbs.pivot_table(index="term", columns="speaker", values="n",
                         aggfunc="sum", fill_value=0.0)
counts = counts[[s for s in main_speakers if s in counts.columns]]
grand_total = counts.values.sum()
col_totals = counts.sum(axis=0)          # tokens per speaker
row_totals = counts.sum(axis=1)          # tokens per term (all speakers)


def keyness(target: str) -> pd.DataFrame:
    a = counts[target].values                      # term freq in target
    c = float(col_totals[target])                  # target tokens
    b = row_totals.values - a                      # term freq in reference
    d = grand_total - c                            # reference tokens
    e1 = c * (a + b) / (c + d)
    e2 = d * (a + b) / (c + d)
    # Expected counts must repartition the observed totals.
    assert np.allclose(e1 + e2, a + b)
    with np.errstate(divide="ignore", invalid="ignore"):
        term_a = np.where(a > 0, a * np.log(a / e1), 0.0)
        term_b = np.where(b > 0, b * np.log(b / e2), 0.0)
    g2 = 2 * (term_a + term_b)
    over = (a / c) > (b / d)                        # over- vs under-used
    out = pd.DataFrame({"term": counts.index, "g2": g2, "over": over,
                        "target_n": a.astype(int)})
    # G2 is a non-negative statistic.
    assert (out["g2"] >= -1e-9).all()
    return out.sort_values("g2", ascending=False)


key = {s: keyness(s) for s in counts.columns}
for s in counts.columns:
    top = key[s][key[s]["over"]].head(8)["term"].tolist()
    print(f"{s} keywords:", ", ".join(top))

Krishna keywords: action, self, sacrifice, knowledge, attachment, intellect, object, being
Arjuna keywords: kill, mouth, family, stand, say, tooth, battle, destruction
Sanjaya keywords: son, blow, king, conch, archer, army, hero, speak


In [10]:
# Figure: top over-used keywords per speaker (log-likelihood).
TOPK = 12
frames = []
for s in counts.columns:
    d = key[s][key[s]["over"]].head(TOPK).copy()
    d["speaker"] = s
    frames.append(d)
key_top = pd.concat(frames, ignore_index=True)
fig_key = px.bar(
    key_top.sort_values("g2"), x="g2", y="term", orientation="h",
    facet_col="speaker", facet_col_spacing=0.12,
    color="speaker", color_discrete_map=SPEAKER_COLOR,
    labels=dict(g2="log-likelihood G²", term=""),
    title="Signature words: content lemmas over-represented in each voice (Dunning G²)",
)
fig_key.update_yaxes(matches=None, showticklabels=True, tickfont=dict(size=10))
fig_key.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_key.update_layout(height=560, showlegend=False)
export(fig_key, "speaker_keyness.html")
fig_key

wrote gita-knowledge-graph/exports/speaker_keyness.html


## 5. Lexical profile

In [11]:
# Content-word density (terms per verse) and vocabulary spread (types) per voice.
tokens = col_totals.rename("content_tokens")
types = (counts > 0).sum(axis=0).rename("distinct_lemmas")
verses = share.set_index("speaker")["verses"]
prof = pd.concat([verses, tokens, types], axis=1).dropna()
prof["tokens_per_verse"] = prof["content_tokens"] / prof["verses"]
# Type-token ratio is length-sensitive; shown with that caveat, not over-read.
prof["ttr"] = prof["distinct_lemmas"] / prof["content_tokens"]
prof = prof.loc[[s for s in main_speakers if s in prof.index]]
print(prof.round(3).to_string())

fig_lex = px.bar(
    prof.reset_index().rename(columns={"index": "speaker"}),
    x="speaker", y="tokens_per_verse", color="speaker",
    color_discrete_map=SPEAKER_COLOR,
    title="Content-word density: content lemmas per verse",
    labels=dict(tokens_per_verse="content lemmas / verse", speaker=""),
)
fig_lex.update_layout(height=380, showlegend=False)
export(fig_lex, "speaker_lexical_density.html")
fig_lex

         verses  content_tokens  distinct_lemmas  tokens_per_verse    ttr
speaker                                                                  
Krishna     574          5467.0           1008.0             9.524  0.184
Arjuna       86           948.0            349.0            11.023  0.368
Sanjaya      40           338.0            156.0             8.450  0.462
wrote gita-knowledge-graph/exports/speaker_lexical_density.html


## 6. Close

In [12]:
gds.close()
print("closed GDS session.")
print("figures written to", EXPORTS.relative_to(ROOT))

closed GDS session.
figures written to gita-knowledge-graph/exports
